<a href="https://colab.research.google.com/github/muratdelen/TAPETUM/blob/main/TAPETUM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title TAPETUM Driverdan yükle

# =========================
# DRIVE → COLAB KOPYALA (TEK KOD)
# =========================

from google.colab import drive
import os, shutil

# 1) Drive bağla
drive.mount('/content/drive')

SRC = "/content/drive/MyDrive/TAPETUM"
DST = "/content/TAPETUM"

#SRC = "/content/drive/MyDrive/TAPETUM/Dataset"
#DST = "/content/TAPETUM/Dataset"

# 2) Hedef varsa sil (çakışma olmasın)
if os.path.exists(DST):
    shutil.rmtree(DST)
    print("Hedef Dosya Temizlendi.")

# 3) Kopyala
print("Hedef dosyadan kopyalama başlatıldı.")
shutil.copytree(SRC, DST)
# 4) Çalışma dizinine geç
os.chdir(DST)

print("✅ Kopyalama tamamlandı")
print("Kaynak :", SRC)
print("Hedef  :", DST)
print("PWD    :", os.getcwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hedef Dosya Temizlendi.
Hedef dosyadan kopyalama başlatıldı.
✅ Kopyalama tamamlandı
Kaynak : /content/drive/MyDrive/TAPETUM
Hedef  : /content/TAPETUM
PWD    : /content/TAPETUM


In [ ]:
#@title tüm kodu çalıştır.

!python /content/TAPETUM/run_all_tapetum_models_colab.py

Görüntülenen çıkış son 5000 satıra kısaltıldı.
Train: 100%|██████████| 86/86 [00:30<00:00,  2.82it/s, loss=0.4422, decomp=0.1408, sEnh=0.0179, lam=1.201, gR=1.176, gG=1.175, gB=1.174]
                                                                                                                                        

Val: 100%|██████████| 100/100 [00:04<00:00, 21.36it/s, loss=0.2969, psnr=23.14, decomp=0.1373, sEnh=0.0214, lam=1.201, gR=1.176, gG=1.175, gB=1.174]
                                                                                                                                                    
Epoch 089 | train_loss 0.3993 | val_loss 0.3773 | val_psnr 18.87 | decomp 0.1683 | smooth_enh 0.0254 | lambda 1.2012 | gate=[1.1756, 1.1753, 1.1737]

Train: 100%|██████████| 86/86 [00:31<00:00,  2.82it/s, loss=0.3600, decomp=0.1342, sEnh=0.0174, lam=1.202, gR=1.176, gG=1.176, gB=1.174]
                                                                                             

In [ ]:
# ============================================================
# @title RUN ALL TAPETUM MODELS
# RetinexNet hariç
# ============================================================

import os
import re
import subprocess

ROOT = "/content/TAPETUM"

MODELS = {
    "RetinexTapetum": os.path.join(ROOT, "RetinexTapetum"),
    "RetinexTapetumRGB": os.path.join(ROOT, "RetinexTapetumRGB"),
    "DecomNetRetinexTapetum": os.path.join(ROOT, "DecomNetRetinexTapetum"),
    "DecomNetRetinexTapetumRGB": os.path.join(ROOT, "DecomNetRetinexTapetumRGB"),
}

TARGET_EPOCHS = 100

def update_config_epochs(config_path, new_epochs=10):
    if not os.path.exists(config_path):
        print("config.py bulunamadı:", config_path)
        return False

    with open(config_path, "r", encoding="utf-8") as f:
        text = f.read()

    new_text, count = re.subn(
        r"^EPOCHS\s*=\s*\d+",
        f"EPOCHS = {new_epochs}",
        text,
        flags=re.MULTILINE
    )

    if count == 0:
        print("EPOCHS satırı bulunamadı:", config_path)
        return False

    with open(config_path, "w", encoding="utf-8") as f:
        f.write(new_text)

    return True

print("===================================================")
print("Running TAPETUM Models")
print("Target Epoch:", TARGET_EPOCHS)
print("===================================================")

for model_name, model_dir in MODELS.items():
    train_script = os.path.join(model_dir, "train.py")
    test_script = os.path.join(model_dir, "test.py")
    config_path = os.path.join(model_dir, "config.py")

    print("\n===================================")
    print("MODEL:", model_name)
    print("DIR  :", model_dir)
    print("===================================")

    if not os.path.exists(model_dir):
        print("Klasör bulunamadı:", model_dir)
        continue

    if not os.path.exists(train_script):
        print("train.py bulunamadı:", train_script)
        continue

    if not os.path.exists(config_path):
        print("config.py bulunamadı:", config_path)
        continue

    ok = update_config_epochs(config_path, TARGET_EPOCHS)
    if not ok:
        print("EPOCHS güncellenemedi, model atlandı.")
        continue

    print("config.py -> EPOCHS =", TARGET_EPOCHS)

    print("TRAIN START")
    subprocess.run(["python", train_script], cwd=model_dir)

    if os.path.exists(test_script):
        print("TEST START")
        subprocess.run(["python", test_script], cwd=model_dir)
    else:
        print("test.py bulunamadı:", test_script)

    print("MODEL FINISHED:", model_name)

print("\n===================================================")
print("ALL MODELS FINISHED")
print("===================================================")

Running TAPETUM Models
Target Epoch: 100

MODEL: RetinexTapetum
DIR  : /content/TAPETUM/RetinexTapetum
config.py -> EPOCHS = 100
TRAIN START


KeyboardInterrupt: 

In [ ]:
# ============================================================
# @title TRAIN ALL TAPETUM MODELS
# ============================================================

import os
import sys
import subprocess
import time
from datetime import datetime

ROOT = "/content/TAPETUM"

MODELS = {
    "RetinexTapetum": os.path.join(ROOT, "RetinexTapetum"),
    "RetinexTapetumRGB": os.path.join(ROOT, "RetinexTapetumRGB"),
    "DecomNetRetinexTapetum": os.path.join(ROOT, "DecomNetRetinexTapetum"),
    "DecomNetRetinexTapetumRGB": os.path.join(ROOT, "DecomNetRetinexTapetumRGB"),
}

LOG_DIR = os.path.join(ROOT, "train_logs")
os.makedirs(LOG_DIR, exist_ok=True)


def is_progress(line):
    """
    tqdm progress bar satırlarını yakalar
    """
    if "%|" in line:
        return True
    if "it/s" in line:
        return True
    if "s/it" in line:
        return True
    return False


print("===================================================")
print("TRAIN START")
print("Time:", datetime.now())
print("===================================================")

os.system("nvidia-smi")

for model_name, model_dir in MODELS.items():

    train_script = os.path.join(model_dir, "train.py")

    print("\n===================================")
    print("MODEL:", model_name)
    print("DIR  :", model_dir)
    print("===================================")

    if not os.path.exists(train_script):
        print("train.py bulunamadı:", train_script)
        continue

    start = time.time()

    log_path = os.path.join(LOG_DIR, f"{model_name}_train.log")

    with open(log_path, "w") as log:

        process = subprocess.Popen(
            [sys.executable, train_script],
            cwd=model_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )

        for line in process.stdout:

            if is_progress(line):
                continue

            line = line.rstrip()
            print(line)
            log.write(line + "\n")

        process.wait()

    elapsed = time.time() - start

    print("\nMODEL TRAIN FINISHED:", model_name)
    print("Elapsed:", round(elapsed,2),"sec")
    print("Log:", log_path)

print("\n===================================================")
print("ALL TRAIN FINISHED")
print("===================================================")

TRAIN START
Time: 2026-03-14 12:46:16.814368

MODEL: RetinexTapetum
DIR  : /content/TAPETUM/RetinexTapetum
[Dataset] ../datasets/LoLv2/LOL-v2/Real_captured/Train/Low
Paired samples: 689
[Dataset] ../datasets/LoLv2/LOL-v2/Real_captured/Test/Low
Paired samples: 100

===== TRAIN START =====


Epoch 1 | VAL PSNR: 12.2071
BEST MODEL SAVED


Epoch 2 | VAL PSNR: 12.2378
BEST MODEL SAVED


Epoch 3 | VAL PSNR: 12.2672
BEST MODEL SAVED


Epoch 4 | VAL PSNR: 12.2959
BEST MODEL SAVED


Epoch 5 | VAL PSNR: 12.3249
BEST MODEL SAVED


Epoch 6 | VAL PSNR: 12.3537
BEST MODEL SAVED


Epoch 7 | VAL PSNR: 12.3823
BEST MODEL SAVED


Epoch 8 | VAL PSNR: 12.4112
BEST MODEL SAVED


Epoch 9 | VAL PSNR: 12.4397
BEST MODEL SAVED


Epoch 10 | VAL PSNR: 12.4690
BEST MODEL SAVED


Epoch 11 | VAL PSNR: 12.4981
BEST MODEL SAVED


Epoch 12 | VAL PSNR: 12.5281
BEST MODEL SAVED


Epoch 13 | VAL PSNR: 12.5570
BEST MODEL SAVED


Epoch 14 | VAL PSNR: 12.5868
BEST MODEL SAVED


Epoch 15 | VAL PSNR: 12.6168
BEST MODEL SAVED


In [ ]:
# ============================================================
# @title TEST ALL TAPETUM MODELS
# ============================================================

import os
import sys
import subprocess
import time

ROOT = "/content/TAPETUM"

MODELS = {
    "RetinexTapetum": os.path.join(ROOT, "RetinexTapetum"),
    "RetinexTapetumRGB": os.path.join(ROOT, "RetinexTapetumRGB"),
    "DecomNetRetinexTapetum": os.path.join(ROOT, "DecomNetRetinexTapetum"),
    "DecomNetRetinexTapetumRGB": os.path.join(ROOT, "DecomNetRetinexTapetumRGB"),
}

LOG_DIR = os.path.join(ROOT, "test_logs")
os.makedirs(LOG_DIR, exist_ok=True)


print("===================================================")
print("TEST START")
print("===================================================")

for model_name, model_dir in MODELS.items():

    test_script = os.path.join(model_dir, "test.py")

    print("\n===================================")
    print("MODEL:", model_name)
    print("DIR  :", model_dir)
    print("===================================")

    if not os.path.exists(test_script):
        print("test.py bulunamadı:", test_script)
        continue

    start = time.time()

    log_path = os.path.join(LOG_DIR, f"{model_name}_test.log")

    with open(log_path, "w") as log:

        process = subprocess.Popen(
            [sys.executable, test_script],
            cwd=model_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )

        for line in process.stdout:
            print(line.rstrip())
            log.write(line)

        process.wait()

    elapsed = time.time() - start

    print("\nMODEL TEST FINISHED:", model_name)
    print("Elapsed:", round(elapsed,2),"sec")
    print("Log:", log_path)

print("\n===================================================")
print("ALL TEST FINISHED")
print("===================================================")

In [ ]:
#@title evaluate_all_models_updated.py çalıştır ve karşılaştırma metni oluşturur

# =========================================================
# COLAB RUNNER FOR /content/TAPETUM/Metrics/evaluate_all_models_updated.py
# =========================================================

import os
import sys
import subprocess

SCRIPT_PATH = "/content/TAPETUM/Metrics/evaluate_all_models_updated.py"
PROJECT_ROOT = "/content/TAPETUM"

# 1) Temel kontrol
if not os.path.exists(PROJECT_ROOT):
    raise FileNotFoundError(f"Proje klasörü bulunamadı: {PROJECT_ROOT}")

if not os.path.exists(SCRIPT_PATH):
    raise FileNotFoundError(f"Script bulunamadı: {SCRIPT_PATH}")

print("✅ Proje klasörü:", PROJECT_ROOT)
print("✅ Script bulundu:", SCRIPT_PATH)

# 2) Çalışma dizinini ayarla
os.chdir(PROJECT_ROOT)
print("📂 Current working directory:", os.getcwd())

# 3) GPU bilgisi
print("\n===== GPU INFO =====")
os.system("nvidia-smi")

# 4) İsteğe bağlı klasör kontrolü
check_paths = [
    "/content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Test/Low",
    "/content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Test/Normal",
    "/content/TAPETUM/RetinexNet/test_results_lolv2",
    "/content/TAPETUM/LoLv2/retinex-tapetum/results/Test",
    "/content/TAPETUM/LoLv2/RetinexTapetumRGB/results/Test",
    "/content/TAPETUM/LoLv2/DecomNetRetinexTapetum/results/Test",
    "/content/TAPETUM/LoLv2/DecomNetRetinexTapetumRGB/results/Test",
]

print("\n===== PATH CHECK =====")
for p in check_paths:
    print(("✅ VAR   " if os.path.exists(p) else "❌ YOK   "), p)

# 5) Scripti çalıştır
print("\n===== SCRIPT RUN =====\n")
result = subprocess.run([sys.executable, SCRIPT_PATH], cwd=PROJECT_ROOT)

# 6) Sonuç
print("\n===== FINISHED =====")
if result.returncode == 0:
    print("✅ Script başarıyla tamamlandı.")
    print("📁 Üretilen tablolar:", "/content/TAPETUM/Metrics/tables")
    print("🖼️ Üretilen görseller:", "/content/TAPETUM/Metrics/visuals")
else:
    print(f"❌ Script hata ile bitti. Return code: {result.returncode}")

✅ Proje klasörü: /content/TAPETUM
✅ Script bulundu: /content/TAPETUM/Metrics/evaluate_all_models_updated.py
📂 Current working directory: /content/TAPETUM

===== GPU INFO =====

===== PATH CHECK =====
✅ VAR    /content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Test/Low
✅ VAR    /content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Test/Normal
✅ VAR    /content/TAPETUM/RetinexNet/test_results_lolv2
✅ VAR    /content/TAPETUM/LoLv2/retinex-tapetum/results/Test
✅ VAR    /content/TAPETUM/LoLv2/RetinexTapetumRGB/results/Test
✅ VAR    /content/TAPETUM/LoLv2/DecomNetRetinexTapetum/results/Test
✅ VAR    /content/TAPETUM/LoLv2/DecomNetRetinexTapetumRGB/results/Test

===== SCRIPT RUN =====


===== FINISHED =====
✅ Script başarıyla tamamlandı.
📁 Üretilen tablolar: /content/TAPETUM/Metrics/tables
🖼️ Üretilen görseller: /content/TAPETUM/Metrics/visuals


In [ ]:
#@title TAPETUM → DRIVE SENKRON KAYIT

# =========================
# COLAB → DRIVE SYNC
# İstenilen klasörleri silip güncelle
# =========================

# 1️⃣ Drive bağla
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

# 2️⃣ Kaynak ve hedef klasör
source_dir = "/content/TAPETUM"
target_dir = "/content/drive/MyDrive/TAPETUM"

# 3️⃣ Drive klasörü yoksa oluştur
os.makedirs(target_dir, exist_ok=True)

# 4️⃣ Silmek istediğin klasörleri yaz
DELETE_FOLDERS = [
    "LoLv2",
    "results",
    "metrics"
]

print("🗑️ Seçilen klasörler siliniyor...")

for folder in DELETE_FOLDERS:
    path = os.path.join(target_dir, folder)
    if os.path.exists(path):
        shutil.rmtree(path)
        print("Silindi:", folder)

print("📂 Dosyalar kopyalanıyor...")

# 5️⃣ TAPETUM klasörünü Drive'a güncelle
for item in os.listdir(source_dir):

    src = os.path.join(source_dir, item)
    dst = os.path.join(target_dir, item)

    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

print("✅ TAPETUM klasörü Drive ile senkronlandı")
print("📁 Konum:", target_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🗑️ Seçilen klasörler siliniyor...
📂 Dosyalar kopyalanıyor...
✅ TAPETUM klasörü Drive ile senkronlandı
📁 Konum: /content/drive/MyDrive/TAPETUM


In [ ]:
# @title RETINEXNET TRAIN

# =========================================================
# RETINEXNET TRAIN - COLAB SINGLE CELL
# WORKDIR : /content/TAPETUM/RetinexNet
# DATASET : /content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured
# =========================================================

%cd /content/TAPETUM/RetinexNet

# 1) Kurulum
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

!pip uninstall -y tensorflow tensorflow-cpu keras tf-keras protobuf -q
!pip install -q \
    "protobuf<7,>=5.26.1" \
    "tensorflow==2.19.1" \
    "tf-keras==2.19.0" \
    pillow numpy

# 2) Yol kontrolü
from glob import glob

PROJECT_DIR = "/content/TAPETUM/RetinexNet"
DATA_ROOT   = "/content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured"

TRAIN_LOW_DIR  = os.path.join(DATA_ROOT, "Train", "Low")
TRAIN_HIGH_DIR = os.path.join(DATA_ROOT, "Train", "Normal")
TEST_LOW_DIR   = os.path.join(DATA_ROOT, "Test", "Low")
TEST_HIGH_DIR  = os.path.join(DATA_ROOT, "Test", "Normal")

print("PROJECT_DIR   :", PROJECT_DIR, os.path.exists(PROJECT_DIR))
print("DATA_ROOT     :", DATA_ROOT, os.path.exists(DATA_ROOT))
print("TRAIN_LOW_DIR :", TRAIN_LOW_DIR, os.path.exists(TRAIN_LOW_DIR))
print("TRAIN_HIGH_DIR:", TRAIN_HIGH_DIR, os.path.exists(TRAIN_HIGH_DIR))
print("TEST_LOW_DIR  :", TEST_LOW_DIR, os.path.exists(TEST_LOW_DIR))
print("TEST_HIGH_DIR :", TEST_HIGH_DIR, os.path.exists(TEST_HIGH_DIR))

train_low_count  = len(glob(os.path.join(TRAIN_LOW_DIR, "*.*")))
train_high_count = len(glob(os.path.join(TRAIN_HIGH_DIR, "*.*")))
test_low_count   = len(glob(os.path.join(TEST_LOW_DIR, "*.*")))
test_high_count  = len(glob(os.path.join(TEST_HIGH_DIR, "*.*")))

print("Train Low count :", train_low_count)
print("Train High count:", train_high_count)
print("Test Low count  :", test_low_count)
print("Test High count :", test_high_count)

assert train_low_count > 0, "Train/Low boş."
assert train_high_count > 0, "Train/Normal boş."
assert test_low_count > 0, "Test/Low boş."
assert test_high_count > 0, "Test/Normal boş."
assert train_low_count == train_high_count, "Train Low ve Train Normal sayıları eşit değil."

# 3) model.py patch
from pathlib import Path

model_path = Path("/content/TAPETUM/RetinexNet/model.py")
model_txt = model_path.read_text(encoding="utf-8")

if "import tensorflow.compat.v1 as tf" not in model_txt:
    model_txt = model_txt.replace(
        "import tensorflow as tf",
        "import os\nos.environ['TF_USE_LEGACY_KERAS'] = '1'\nimport tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()"
    )

bad_shuffle = """tmp = list(zip(train_low_data, train_high_data))
                        random.shuffle(list(tmp))
                        train_low_data, train_high_data  = zip(*tmp)"""

good_shuffle = """tmp = list(zip(train_low_data, train_high_data))
                        random.shuffle(tmp)
                        train_low_data, train_high_data = zip(*tmp)"""

model_txt = model_txt.replace(bad_shuffle, good_shuffle)

model_txt = model_txt.replace(
    "with tf.variable_scope('RelightNet'):",
    "with tf.variable_scope('RelightNet', reuse=tf.AUTO_REUSE):"
)

model_path.write_text(model_txt, encoding="utf-8")
print("model.py patch tamam.")

# 4) Sürüm kontrolü
import tensorflow as tf
import tf_keras
print("TensorFlow version:", tf.__version__)
print("tf_keras version  :", tf_keras.__version__)

# 5) Train
!python main.py \
  --phase=train \
  --use_gpu=1 \
  --gpu_idx=0 \
  --gpu_mem=0.9 \
  --epoch=100 \
  --batch_size=16 \
  --patch_size=48 \
  --start_lr=0.001 \
  --eval_every_epoch=20 \
  --checkpoint_dir=./checkpoint \
  --sample_dir=./sample

# 6) Kontrol
print("\n===== CHECKPOINT FILES =====")
!find /content/TAPETUM/RetinexNet/checkpoint -maxdepth 3 -type f | sort

print("\n===== SAMPLE FILES =====")
!find /content/TAPETUM/RetinexNet/sample -maxdepth 3 -type f | sort | tail -30

Görüntülenen çıkış son 5000 satıra kısaltıldı.
Decom Epoch: [86] [  29/  43] time: 198.2218, loss: 0.023186
Decom Epoch: [86] [  30/  43] time: 198.2539, loss: 0.025316
Decom Epoch: [86] [  31/  43] time: 198.2868, loss: 0.027138
Decom Epoch: [86] [  32/  43] time: 198.3202, loss: 0.027457
Decom Epoch: [86] [  33/  43] time: 198.3541, loss: 0.029908
Decom Epoch: [86] [  34/  43] time: 198.3876, loss: 0.024125
Decom Epoch: [86] [  35/  43] time: 198.4223, loss: 0.030577
Decom Epoch: [86] [  36/  43] time: 198.4589, loss: 0.031661
Decom Epoch: [86] [  37/  43] time: 198.4918, loss: 0.027962
Decom Epoch: [86] [  38/  43] time: 198.5251, loss: 0.026153
Decom Epoch: [86] [  39/  43] time: 198.5589, loss: 0.025256
Decom Epoch: [86] [  40/  43] time: 198.5918, loss: 0.036063
Decom Epoch: [86] [  41/  43] time: 198.6258, loss: 0.028814
Decom Epoch: [86] [  42/  43] time: 198.6608, loss: 0.034631
Decom Epoch: [86] [  43/  43] time: 198.6940, loss: 0.026879
Decom Epoch: [87] [   1/  43] time: 19

In [ ]:
# @title RETINEXNET TEST

# @title RETINEXNET TEST + METRICS

# =========================================================
# RETINEXNET TEST - COLAB SINGLE CELL
# WORKDIR : /content/TAPETUM/RetinexNet
# DATASET : /content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured
# =========================================================

%cd /content/TAPETUM/RetinexNet

# 1) Kurulum
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

!pip uninstall -y tensorflow tensorflow-cpu keras tf-keras protobuf -q
!pip install -q \
    "protobuf<7,>=5.26.1" \
    "tensorflow==2.19.1" \
    "tf-keras==2.19.0" \
    pillow numpy pandas scikit-image opencv-python-headless

# 2) Yol tanımları
from glob import glob
from pathlib import Path
import shutil

PROJECT_DIR = "/content/TAPETUM/RetinexNet"
DATA_ROOT   = "/content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured"

TEST_LOW_DIR   = os.path.join(DATA_ROOT, "Test", "Low")
TEST_HIGH_DIR  = os.path.join(DATA_ROOT, "Test", "Normal")
SAVE_DIR       = os.path.join(PROJECT_DIR, "test_results_lolv2")
MODEL_DECOM    = os.path.join(PROJECT_DIR, "model", "Decom")
MODEL_RELIGHT  = os.path.join(PROJECT_DIR, "model", "Relight")
CKPT_DECOM     = os.path.join(PROJECT_DIR, "checkpoint", "Decom")
CKPT_RELIGHT   = os.path.join(PROJECT_DIR, "checkpoint", "Relight")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(MODEL_DECOM, exist_ok=True)
os.makedirs(MODEL_RELIGHT, exist_ok=True)

print("PROJECT_DIR   :", PROJECT_DIR, os.path.exists(PROJECT_DIR))
print("TEST_LOW_DIR  :", TEST_LOW_DIR, os.path.exists(TEST_LOW_DIR))
print("TEST_HIGH_DIR :", TEST_HIGH_DIR, os.path.exists(TEST_HIGH_DIR))
print("SAVE_DIR      :", SAVE_DIR)

print("Test Low count :", len(glob(os.path.join(TEST_LOW_DIR, "*.*"))))
print("Test High count:", len(glob(os.path.join(TEST_HIGH_DIR, "*.*"))))

# 3) model.py patch
model_path = Path(os.path.join(PROJECT_DIR, "model.py"))
model_txt = model_path.read_text(encoding="utf-8")

if "import tensorflow.compat.v1 as tf" not in model_txt:
    model_txt = model_txt.replace(
        "import tensorflow as tf",
        "import os\nos.environ['TF_USE_LEGACY_KERAS'] = '1'\nimport tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()"
    )

bad_shuffle = """tmp = list(zip(train_low_data, train_high_data))
                        random.shuffle(list(tmp))
                        train_low_data, train_high_data  = zip(*tmp)"""

good_shuffle = """tmp = list(zip(train_low_data, train_high_data))
                        random.shuffle(tmp)
                        train_low_data, train_high_data = zip(*tmp)"""

model_txt = model_txt.replace(bad_shuffle, good_shuffle)
model_txt = model_txt.replace(
    "with tf.variable_scope('RelightNet'):",
    "with tf.variable_scope('RelightNet', reuse=tf.AUTO_REUSE):"
)

model_path.write_text(model_txt, encoding="utf-8")
print("model.py patch tamam.")

# 4) Checkpoint -> model kopyala
for src_dir, dst_dir in [(CKPT_DECOM, MODEL_DECOM), (CKPT_RELIGHT, MODEL_RELIGHT)]:
    if os.path.exists(src_dir):
        for f in glob(os.path.join(src_dir, "*")):
            if os.path.isfile(f):
                shutil.copy2(f, os.path.join(dst_dir, os.path.basename(f)))

print("\n===== MODEL FILES =====")
!find /content/TAPETUM/RetinexNet/model -maxdepth 3 -type f | sort

# 5) Sürüm kontrolü
import tensorflow as tf
import tf_keras
print("\nTensorFlow version:", tf.__version__)
print("tf_keras version  :", tf_keras.__version__)

# 6) Test
!python main.py \
  --phase=test \
  --use_gpu=1 \
  --gpu_idx=0 \
  --gpu_mem=0.9 \
  --test_dir=../datasets/LoLv2/LOL-v2/Real_captured/Test/Low \
  --save_dir=./test_results_lolv2 \
  --decom=1

print("\n===== TEST OUTPUT FILES =====")
!find /content/TAPETUM/RetinexNet/test_results_lolv2 -maxdepth 1 -type f | sort | head -100

# 7) Metrik hesaplama
import cv2
import math
import numpy as np
import pandas as pd
from skimage.metrics import structural_similarity as ssim

def calculate_psnr_uint8(img1, img2):
    mse = np.mean((img1.astype(np.float32) - img2.astype(np.float32)) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * math.log10(255.0 / math.sqrt(mse))

def list_images(d):
    exts = (".png", ".jpg", ".jpeg", ".bmp")
    return sorted([f for f in os.listdir(d) if f.lower().endswith(exts)])

gt_files = list_images(TEST_HIGH_DIR)

# Sadece enhanced çıktıların değerlendirilmesi
# RetinexNet test çıktılarında esas sonuç dosyaları genelde "_S.png" ile biter
pred_files = [f for f in list_images(SAVE_DIR) if f.endswith("_S.png")]

print("\nGT count   :", len(gt_files))
print("Pred count :", len(pred_files))

rows = []

for pred_name in pred_files:
    gt_name = pred_name.replace("_S.png", ".png")

    gt_path = os.path.join(TEST_HIGH_DIR, gt_name)
    pred_path = os.path.join(SAVE_DIR, pred_name)

    if not os.path.exists(gt_path):
        print("GT bulunamadı:", gt_name)
        continue

    gt = cv2.imread(gt_path)
    pred = cv2.imread(pred_path)

    if gt is None or pred is None:
        print("Okunamadı:", pred_name)
        continue

    gt = cv2.cvtColor(gt, cv2.COLOR_BGR2RGB)
    pred = cv2.cvtColor(pred, cv2.COLOR_BGR2RGB)

    if gt.shape != pred.shape:
        pred = cv2.resize(pred, (gt.shape[1], gt.shape[0]))

    psnr_val = calculate_psnr_uint8(pred, gt)
    ssim_val = ssim(gt, pred, channel_axis=2, data_range=255)
    mae_val  = np.mean(np.abs(pred.astype(np.float32) - gt.astype(np.float32))) / 255.0
    mse_val  = np.mean((pred.astype(np.float32) - gt.astype(np.float32)) ** 2) / (255.0 ** 2)

    rows.append([
        "retinexnet",
        gt_name,
        pred_name,
        psnr_val,
        ssim_val,
        mae_val,
        mse_val
    ])

df = pd.DataFrame(rows, columns=[
    "model", "image", "pred_file", "psnr", "ssim", "mae", "mse"
])

df = df.sort_values("image").reset_index(drop=True)

summary_df = pd.DataFrame([{
    "model": "retinexnet",
    "image_count": len(df),
    "psnr": df["psnr"].mean() if len(df) else np.nan,
    "ssim": df["ssim"].mean() if len(df) else np.nan,
    "mae":  df["mae"].mean() if len(df) else np.nan,
    "mse":  df["mse"].mean() if len(df) else np.nan
}])

detail_csv  = os.path.join(PROJECT_DIR, "retinexnet_test_metrics.csv")
summary_csv = os.path.join(PROJECT_DIR, "retinexnet_test_metrics_summary.csv")

df.to_csv(detail_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

print("\n===== DETAYLI SONUÇLAR =====")
print(df.head())

print("\n===== ORTALAMA SONUÇLAR =====")
print(summary_df)

print("\nKaydedildi:")
print(detail_csv)
print(summary_csv)

/content/TAPETUM/RetinexNet
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.19.1 which is incompatible.
PROJECT_DIR   : /content/TAPETUM/RetinexNet True
TEST_LOW_DIR  : /content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Test/Low True
TEST_HIGH_DIR : /content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Test/Normal True
SAVE_DIR      : /content/TAPETUM/RetinexNet/test_results_lolv2
Test Low count : 100
Test High count: 100
model.py patch tamam.

===== MODEL FILES =====
/content/TAPETUM/RetinexNet/model/Decom/checkpoint
/content/TAPETUM/RetinexNet/model/Decom/RetinexNet-Decom-1720.data-00000-of-00001
/content/TAPETUM/RetinexNet/model/Decom/RetinexNet-Decom-1720.index
/content/TAPETUM/RetinexNet/model/Decom/RetinexNet-Decom-1720.meta
/content/TAPETUM/RetinexNet/model/Decom/R

In [ ]:
#@title TRAIN

# ============================================================
# 7-MODEL LOW-LIGHT ENHANCEMENT TRAIN CODE (COLAB READY)
# Supports:
# 1) retinexnet_baseline
# 2) tapetum
# 3) tapetum_rgb
# 4) retinex_tapetum
# 5) retinex_tapetum_rgb
# 6) decomnet_retinex_tapetum
# 7) decomnet_retinex_tapetum_rgb
# ============================================================

# =========================
# 1) Imports
# =========================
import os
import math
import json
import random
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image

# =========================
# 2) CONFIG
# =========================
MODEL_NAME = "retinexnet_baseline"   # <- BURAYI DEĞİŞTİR
# seçenekler:
# "retinexnet_baseline"
# "tapetum"
# "tapetum_rgb"
# "retinex_tapetum"
# "retinex_tapetum_rgb"
# "decomnet_retinex_tapetum"
# "decomnet_retinex_tapetum_rgb"

DATA_ROOT = "/content/ADF-LLIE/datasets/LoLv2/LOL-v2/Real_captured"
RUN_ROOT = f"/content/ADF-LLIE/LoLv2/{MODEL_NAME}"

CONFIG = {
    "seed": 42,
    "batch_size": 8,
    "num_workers": 2,
    "crop_size": 256,
    "epochs": 120,
    "lr": 2e-4,
    "weight_decay": 1e-6,
    "grad_clip": 1.0,
    "eta_min": 1e-6,
    "base_channels": 32,
    "lambda_init": 1.0,
    "img_exts": (".png", ".jpg", ".jpeg", ".bmp"),

    # loss weights
    "w_l1": 1.0,
    "w_ssim": 0.5,
    "w_color": 0.1,
    "w_attn": 0.01,

    # decomposition-related optional
    "w_recon_decomp": 0.0,   # istersen sonra artırırsın
}

TRAIN_LOW_DIR  = os.path.join(DATA_ROOT, "Train", "Low")
TRAIN_HIGH_DIR = os.path.join(DATA_ROOT, "Train", "Normal")
VAL_LOW_DIR    = os.path.join(DATA_ROOT, "Test", "Low")
VAL_HIGH_DIR   = os.path.join(DATA_ROOT, "Test", "Normal")

CKPT_DIR = os.path.join(RUN_ROOT, "checkpoints")
LOG_JSON = os.path.join(RUN_ROOT, "history.json")

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RUN_ROOT, exist_ok=True)

# =========================
# 3) Seed
# =========================
def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CONFIG["seed"])

device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)
print("MODEL_NAME:", MODEL_NAME)
print("TRAIN_LOW_DIR :", TRAIN_LOW_DIR)
print("TRAIN_HIGH_DIR:", TRAIN_HIGH_DIR)
print("VAL_LOW_DIR   :", VAL_LOW_DIR)
print("VAL_HIGH_DIR  :", VAL_HIGH_DIR)
print("RUN_ROOT      :", RUN_ROOT)
print("CKPT_DIR      :", CKPT_DIR)

# =========================
# 4) Dataset
# =========================
def list_images(folder):
    return sorted([f for f in os.listdir(folder) if f.lower().endswith(CONFIG["img_exts"])])

class LOLPairDataset(Dataset):
    def __init__(self, low_dir, high_dir, crop_size=256, training=True):
        self.low_dir = low_dir
        self.high_dir = high_dir
        self.crop_size = crop_size
        self.training = training
        self.to_tensor = transforms.ToTensor()

        low_files = list_images(low_dir)
        high_files = set(list_images(high_dir))
        self.files = [f for f in low_files if f in high_files]

        print(f"[Dataset] {low_dir}")
        print(f"Paired samples: {len(self.files)}")

    def __len__(self):
        return len(self.files)

    def _resize_if_needed(self, low, high):
        w, h = low.size
        if w < self.crop_size or h < self.crop_size:
            new_w = max(w, self.crop_size)
            new_h = max(h, self.crop_size)
            low = low.resize((new_w, new_h), Image.BICUBIC)
            high = high.resize((new_w, new_h), Image.BICUBIC)
        return low, high

    def _random_crop(self, low, high):
        w, h = low.size
        x = random.randint(0, w - self.crop_size)
        y = random.randint(0, h - self.crop_size)
        low = low.crop((x, y, x + self.crop_size, y + self.crop_size))
        high = high.crop((x, y, x + self.crop_size, y + self.crop_size))
        return low, high

    def _augment(self, low, high):
        if random.random() < 0.5:
            low = low.transpose(Image.FLIP_LEFT_RIGHT)
            high = high.transpose(Image.FLIP_LEFT_RIGHT)

        if random.random() < 0.5:
            low = low.transpose(Image.FLIP_TOP_BOTTOM)
            high = high.transpose(Image.FLIP_TOP_BOTTOM)

        k = random.randint(0, 3)
        if k:
            low = low.rotate(90 * k)
            high = high.rotate(90 * k)
        return low, high

    def __getitem__(self, idx):
        fname = self.files[idx]
        low = Image.open(os.path.join(self.low_dir, fname)).convert("RGB")
        high = Image.open(os.path.join(self.high_dir, fname)).convert("RGB")

        if self.training:
            low, high = self._resize_if_needed(low, high)
            low, high = self._random_crop(low, high)
            low, high = self._augment(low, high)

        low = self.to_tensor(low)
        high = self.to_tensor(high)

        return {"low": low, "high": high, "name": fname}

# =========================
# 5) Utils
# =========================
def gaussian_kernel(kernel_size=21, sigma=5.0, channels=3, device="cpu"):
    ax = torch.arange(kernel_size, device=device) - kernel_size // 2
    xx, yy = torch.meshgrid(ax, ax, indexing="ij")
    kernel = torch.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    kernel = kernel / kernel.sum()
    kernel = kernel.view(1, 1, kernel_size, kernel_size).repeat(channels, 1, 1, 1)
    return kernel

def gaussian_blur(x, kernel_size=21, sigma=5.0):
    c = x.shape[1]
    kernel = gaussian_kernel(kernel_size=kernel_size, sigma=sigma, channels=c, device=x.device)
    pad = kernel_size // 2
    x = F.pad(x, (pad, pad, pad, pad), mode="reflect")
    out = F.conv2d(x, kernel, groups=c)
    return out

def fixed_retinex_decompose(x):
    L = gaussian_blur(x, kernel_size=21, sigma=5.0)
    R = x / (L + 1e-6)
    R = torch.clamp(R, 0.0, 5.0)
    return R, L

# =========================
# 6) Blocks
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class TinyUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, base=32):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = ConvBlock(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out_conv = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))

        d2 = self.up2(b)
        if d2.shape[-2:] != e2.shape[-2:]:
            d2 = F.interpolate(d2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        if d1.shape[-2:] != e1.shape[-2:]:
            d1 = F.interpolate(d1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out_conv(d1)

class RGBGainHead(nn.Module):
    def __init__(self, in_ch=3, base=32):
        super().__init__()
        self.net = TinyUNet(in_ch=in_ch, out_ch=3, base=base)

    def forward(self, x):
        return torch.sigmoid(self.net(x))

class TapetumAttentionNet(nn.Module):
    def __init__(self, in_ch=3, base=32):
        super().__init__()
        self.net = TinyUNet(in_ch=in_ch, out_ch=3, base=base)

    def forward(self, x):
        raw = self.net(x)
        T = torch.sigmoid(raw) * (1.0 - x)
        return T

class LearnableDecomNet(nn.Module):
    """
    Basit, Colab-ready learnable decomposition.
    x -> [R_raw, L_raw]
    """
    def __init__(self, base=32):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, base, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base, base, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base, base, 3, 1, 1),
            nn.ReLU(inplace=True),
        )
        self.r_head = nn.Conv2d(base, 3, 3, 1, 1)
        self.l_head = nn.Conv2d(base, 3, 3, 1, 1)

    def forward(self, x):
        f = self.backbone(x)
        R = torch.sigmoid(self.r_head(f)) * 5.0
        L = torch.sigmoid(self.l_head(f))
        return R, L

# =========================
# 7) Models
# =========================
class RetinexNetBaseline(nn.Module):
    """
    Fixed decomposition + simple illumination refinement.
    """
    def __init__(self, base=32):
        super().__init__()
        self.illum_net = TinyUNet(in_ch=3, out_ch=3, base=base)

    def forward(self, x):
        R, L = fixed_retinex_decompose(x)
        delta = torch.sigmoid(self.illum_net(L))
        L_t = torch.clamp(L + 0.5 * delta, 0.0, 1.0)
        I_enh = torch.clamp(R * L_t, 0.0, 1.0)
        return {
            "enhanced": I_enh,
            "reflectance": R,
            "illumination": L,
            "illumination_t": L_t,
            "tapetum_attention": torch.zeros_like(x),
            "lambda": torch.tensor(0.0, device=x.device)
        }

class TapetumModel(nn.Module):
    def __init__(self, base=32, lambda_init=1.0):
        super().__init__()
        self.tapetum_net = TapetumAttentionNet(in_ch=3, base=base)
        self.lambda_param = nn.Parameter(torch.tensor(lambda_init, dtype=torch.float32))

    def forward(self, x):
        T = self.tapetum_net(x)
        lam = F.softplus(self.lambda_param)
        I_enh = torch.clamp(x * (1.0 + lam * T), 0.0, 1.0)
        return {
            "enhanced": I_enh,
            "tapetum_attention": T,
            "lambda": lam
        }

class TapetumRGBModel(nn.Module):
    def __init__(self, base=32, lambda_init=1.0):
        super().__init__()
        self.tapetum_net = TapetumAttentionNet(in_ch=3, base=base)
        self.rgb_gain = RGBGainHead(in_ch=3, base=base)
        self.lambda_param = nn.Parameter(torch.tensor(lambda_init, dtype=torch.float32))

    def forward(self, x):
        T = self.tapetum_net(x)
        G = self.rgb_gain(x)  # per-channel gain in [0,1]
        lam = F.softplus(self.lambda_param)
        I_enh = torch.clamp(x * (1.0 + lam * T * G), 0.0, 1.0)
        return {
            "enhanced": I_enh,
            "tapetum_attention": T,
            "rgb_gain": G,
            "lambda": lam
        }

class RetinexTapetumModel(nn.Module):
    def __init__(self, base=32, lambda_init=1.0):
        super().__init__()
        self.tapetum_net = TapetumAttentionNet(in_ch=3, base=base)
        self.lambda_param = nn.Parameter(torch.tensor(lambda_init, dtype=torch.float32))

    def forward(self, x):
        R, L = fixed_retinex_decompose(x)
        T = self.tapetum_net(L)
        lam = F.softplus(self.lambda_param)
        L_t = L * (1.0 + lam * T)
        I_enh = torch.clamp(R * L_t, 0.0, 1.0)
        return {
            "enhanced": I_enh,
            "reflectance": R,
            "illumination": L,
            "illumination_t": L_t,
            "tapetum_attention": T,
            "lambda": lam
        }

class RetinexTapetumRGBModel(nn.Module):
    def __init__(self, base=32, lambda_init=1.0):
        super().__init__()
        self.tapetum_net = TapetumAttentionNet(in_ch=3, base=base)
        self.rgb_gain = RGBGainHead(in_ch=3, base=base)
        self.lambda_param = nn.Parameter(torch.tensor(lambda_init, dtype=torch.float32))

    def forward(self, x):
        R, L = fixed_retinex_decompose(x)
        T = self.tapetum_net(L)
        G = self.rgb_gain(L)
        lam = F.softplus(self.lambda_param)
        L_t = L * (1.0 + lam * T * G)
        I_enh = torch.clamp(R * L_t, 0.0, 1.0)
        return {
            "enhanced": I_enh,
            "reflectance": R,
            "illumination": L,
            "illumination_t": L_t,
            "tapetum_attention": T,
            "rgb_gain": G,
            "lambda": lam
        }

class DecomNetRetinexTapetumModel(nn.Module):
    def __init__(self, base=32, lambda_init=1.0):
        super().__init__()
        self.decom = LearnableDecomNet(base=base)
        self.tapetum_net = TapetumAttentionNet(in_ch=3, base=base)
        self.lambda_param = nn.Parameter(torch.tensor(lambda_init, dtype=torch.float32))

    def forward(self, x):
        R, L = self.decom(x)
        T = self.tapetum_net(L)
        lam = F.softplus(self.lambda_param)
        L_t = L * (1.0 + lam * T)
        I_enh = torch.clamp(R * L_t, 0.0, 1.0)
        recon = torch.clamp(R * L, 0.0, 1.0)
        return {
            "enhanced": I_enh,
            "reconstructed": recon,
            "reflectance": R,
            "illumination": L,
            "illumination_t": L_t,
            "tapetum_attention": T,
            "lambda": lam
        }

class DecomNetRetinexTapetumRGBModel(nn.Module):
    def __init__(self, base=32, lambda_init=1.0):
        super().__init__()
        self.decom = LearnableDecomNet(base=base)
        self.tapetum_net = TapetumAttentionNet(in_ch=3, base=base)
        self.rgb_gain = RGBGainHead(in_ch=3, base=base)
        self.lambda_param = nn.Parameter(torch.tensor(lambda_init, dtype=torch.float32))

    def forward(self, x):
        R, L = self.decom(x)
        T = self.tapetum_net(L)
        G = self.rgb_gain(L)
        lam = F.softplus(self.lambda_param)
        L_t = L * (1.0 + lam * T * G)
        I_enh = torch.clamp(R * L_t, 0.0, 1.0)
        recon = torch.clamp(R * L, 0.0, 1.0)
        return {
            "enhanced": I_enh,
            "reconstructed": recon,
            "reflectance": R,
            "illumination": L,
            "illumination_t": L_t,
            "tapetum_attention": T,
            "rgb_gain": G,
            "lambda": lam
        }

def build_model(model_name, base=32, lambda_init=1.0):
    if model_name == "retinexnet_baseline":
        return RetinexNetBaseline(base=base)
    elif model_name == "tapetum":
        return TapetumModel(base=base, lambda_init=lambda_init)
    elif model_name == "tapetum_rgb":
        return TapetumRGBModel(base=base, lambda_init=lambda_init)
    elif model_name == "retinex_tapetum":
        return RetinexTapetumModel(base=base, lambda_init=lambda_init)
    elif model_name == "retinex_tapetum_rgb":
        return RetinexTapetumRGBModel(base=base, lambda_init=lambda_init)
    elif model_name == "decomnet_retinex_tapetum":
        return DecomNetRetinexTapetumModel(base=base, lambda_init=lambda_init)
    elif model_name == "decomnet_retinex_tapetum_rgb":
        return DecomNetRetinexTapetumRGBModel(base=base, lambda_init=lambda_init)
    else:
        raise ValueError(f"Unknown MODEL_NAME: {model_name}")

# =========================
# 8) Loss
# =========================
def charbonnier_loss(pred, target, eps=1e-3):
    return torch.mean(torch.sqrt((pred - target) ** 2 + eps ** 2))

def create_gaussian_window(window_size, channel, device):
    sigma = 1.5
    coords = torch.arange(window_size, dtype=torch.float32, device=device) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window_1d = g.unsqueeze(1)
    window_2d = window_1d @ window_1d.t()
    window_2d = window_2d.unsqueeze(0).unsqueeze(0)
    return window_2d.expand(channel, 1, window_size, window_size).contiguous()

def ssim_loss(pred, target, window_size=11):
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    channel = pred.size(1)
    window = create_gaussian_window(window_size, channel, pred.device)

    mu1 = F.conv2d(pred, window, padding=window_size // 2, groups=channel)
    mu2 = F.conv2d(target, window, padding=window_size // 2, groups=channel)

    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.conv2d(pred * pred, window, padding=window_size // 2, groups=channel) - mu1_sq
    sigma2_sq = F.conv2d(target * target, window, padding=window_size // 2, groups=channel) - mu2_sq
    sigma12 = F.conv2d(pred * target, window, padding=window_size // 2, groups=channel) - mu1_mu2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / (
        (mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2) + 1e-8
    )
    return 1.0 - ssim_map.mean()

def color_consistency_loss(pred, target):
    pred_mean = pred.mean(dim=[2, 3])
    target_mean = target.mean(dim=[2, 3])
    return F.l1_loss(pred_mean, target_mean)

def attention_regularization(T):
    return T.mean()

def total_loss_fn(output, gt, low_input, model_name):
    pred = output["enhanced"]

    l1 = charbonnier_loss(pred, gt)
    ssim_l = ssim_loss(pred, gt)
    color_l = color_consistency_loss(pred, gt)

    if "tapetum_attention" in output:
        attn_l = attention_regularization(output["tapetum_attention"])
    else:
        attn_l = torch.tensor(0.0, device=gt.device)

    total = (
        CONFIG["w_l1"] * l1 +
        CONFIG["w_ssim"] * ssim_l +
        CONFIG["w_color"] * color_l +
        CONFIG["w_attn"] * attn_l
    )

    # optional learnable decomposition consistency
    recon_l = torch.tensor(0.0, device=gt.device)
    if "reconstructed" in output and CONFIG["w_recon_decomp"] > 0:
        recon_l = F.l1_loss(output["reconstructed"], low_input)
        total = total + CONFIG["w_recon_decomp"] * recon_l

    logs = {
        "total": total.item(),
        "l1": l1.item(),
        "ssim": ssim_l.item(),
        "color": color_l.item(),
        "attn": attn_l.item(),
        "recon_decomp": recon_l.item(),
    }
    return total, logs

def calc_psnr(pred, target):
    mse = F.mse_loss(pred, target).item()
    if mse == 0:
        return 100.0
    return 10.0 * math.log10(1.0 / mse)

# =========================
# 9) DataLoaders
# =========================
train_dataset = LOLPairDataset(
    low_dir=TRAIN_LOW_DIR,
    high_dir=TRAIN_HIGH_DIR,
    crop_size=CONFIG["crop_size"],
    training=True
)

val_dataset = LOLPairDataset(
    low_dir=VAL_LOW_DIR,
    high_dir=VAL_HIGH_DIR,
    crop_size=CONFIG["crop_size"],
    training=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))

# =========================
# 10) Build model
# =========================
model = build_model(
    MODEL_NAME,
    base=CONFIG["base_channels"],
    lambda_init=CONFIG["lambda_init"]
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG["lr"],
    betas=(0.9, 0.999),
    weight_decay=CONFIG["weight_decay"]
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG["epochs"],
    eta_min=CONFIG["eta_min"]
)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable params:", f"{num_params:,}")

# =========================
# 11) Train / Val
# =========================
def train_one_epoch(model, loader, optimizer, device, model_name):
    model.train()
    running = {
        "total": 0.0,
        "l1": 0.0,
        "ssim": 0.0,
        "color": 0.0,
        "attn": 0.0,
        "recon_decomp": 0.0,
        "psnr": 0.0,
    }

    pbar = tqdm(loader, total=len(loader), desc="Train", leave=False)
    for batch in pbar:
        low = batch["low"].to(device, non_blocking=True)
        high = batch["high"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        output = model(low)
        loss, logs = total_loss_fn(output, high, low, model_name)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        optimizer.step()

        psnr = calc_psnr(output["enhanced"].detach(), high)

        for k in logs:
            running[k] += logs[k]
        running["psnr"] += psnr

        lam_val = output["lambda"].item() if "lambda" in output else 0.0
        pbar.set_postfix({
            "loss": f"{logs['total']:.4f}",
            "psnr": f"{psnr:.2f}",
            "lam": f"{lam_val:.3f}"
        })

    n = len(loader)
    for k in running:
        running[k] /= n
    return running

@torch.no_grad()
def validate(model, loader, device, model_name):
    model.eval()
    running = {
        "total": 0.0,
        "l1": 0.0,
        "ssim": 0.0,
        "color": 0.0,
        "attn": 0.0,
        "recon_decomp": 0.0,
        "psnr": 0.0,
    }

    pbar = tqdm(loader, total=len(loader), desc="Val", leave=False)
    for batch in pbar:
        low = batch["low"].to(device, non_blocking=True)
        high = batch["high"].to(device, non_blocking=True)

        output = model(low)
        loss, logs = total_loss_fn(output, high, low, model_name)
        psnr = calc_psnr(output["enhanced"], high)

        for k in logs:
            running[k] += logs[k]
        running["psnr"] += psnr

        lam_val = output["lambda"].item() if "lambda" in output else 0.0
        pbar.set_postfix({
            "loss": f"{logs['total']:.4f}",
            "psnr": f"{psnr:.2f}",
            "lam": f"{lam_val:.3f}"
        })

    n = len(loader)
    for k in running:
        running[k] /= n
    return running

# =========================
# 12) Training loop
# =========================
best_psnr = -1.0
history = []

print("\n===== TRAIN START =====")
for epoch in range(1, CONFIG["epochs"] + 1):
    train_logs = train_one_epoch(model, train_loader, optimizer, device, MODEL_NAME)
    val_logs = validate(model, val_loader, device, MODEL_NAME)
    scheduler.step()

    current_lambda = (
        F.softplus(model.lambda_param).item()
        if hasattr(model, "lambda_param")
        else 0.0
    )

    log_str = (
        f"Epoch {epoch:03d} | "
        f"train_loss {train_logs['total']:.4f} | "
        f"train_psnr {train_logs['psnr']:.2f} | "
        f"val_loss {val_logs['total']:.4f} | "
        f"val_psnr {val_logs['psnr']:.2f} | "
        f"lambda {current_lambda:.4f}"
    )
    print(log_str)

    history.append({
        "epoch": epoch,
        "train": train_logs,
        "val": val_logs,
        "lambda": current_lambda
    })

    with open(LOG_JSON, "w") as f:
        json.dump(history, f, indent=2)

    ckpt = {
        "epoch": epoch,
        "model_name": MODEL_NAME,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_psnr": best_psnr,
        "history": history,
        "config": CONFIG
    }

    torch.save(ckpt, os.path.join(CKPT_DIR, "last.pth"))

    if val_logs["psnr"] > best_psnr:
        best_psnr = val_logs["psnr"]
        ckpt["best_psnr"] = best_psnr
        torch.save(ckpt, os.path.join(CKPT_DIR, "best.pth"))
        print(f"✅ Best model updated -> PSNR: {best_psnr:.4f}")

print("===== TRAIN FINISHED =====")
print("Best PSNR:", best_psnr)
print("Best checkpoint:", os.path.join(CKPT_DIR, "best.pth"))
print("Last checkpoint:", os.path.join(CKPT_DIR, "last.pth"))
print("History JSON:", LOG_JSON)

DEVICE: cuda
MODEL_NAME: retinexnet_baseline
TRAIN_LOW_DIR : /content/ADF-LLIE/datasets/LoLv2/LOL-v2/Real_captured/Train/Low
TRAIN_HIGH_DIR: /content/ADF-LLIE/datasets/LoLv2/LOL-v2/Real_captured/Train/Normal
VAL_LOW_DIR   : /content/ADF-LLIE/datasets/LoLv2/LOL-v2/Real_captured/Test/Low
VAL_HIGH_DIR  : /content/ADF-LLIE/datasets/LoLv2/LOL-v2/Real_captured/Test/Normal
RUN_ROOT      : /content/ADF-LLIE/LoLv2/retinexnet_baseline
CKPT_DIR      : /content/ADF-LLIE/LoLv2/retinexnet_baseline/checkpoints


FileNotFoundError: [Errno 2] No such file or directory: '/content/ADF-LLIE/datasets/LoLv2/LOL-v2/Real_captured/Train/Low'